# Replay του 1ου γύρου: ποιο κομμάτι του εχθρικού μηνύματος κάνει τη ζημιά;

**Τι ξέρουμε.** Στα κελιά πηγής βλάβης (αστέρας, PD, `framing_competitive`) η
βλάβη έχει δύο ανεξάρτητους δρόμους. Ο **γράφων** που ξαναβλέπει το δικό του
εχθρικό μήνυμα συνεργάζεται λιγότερο, ακόμα κι όταν δεν παραδίδεται τίποτα
(`block_all`). Οι **αναγνώστες** που λαμβάνουν εχθρικά μηνύματα συνεργάζονται
λιγότερο, ακόμα κι όταν το δικό τους μήνυμα κρύβεται (`hideown`).

**Τι ρωτάμε.** Κάθε κλήση του μοντέλου είναι stateless και ο 1ος γύρος δεν έχει
ιστορικό. Άρα μια καταγεγραμμένη κλήση απόφασης του 1ου γύρου ξαναχτίζεται
ακριβώς και ξαναρωτιέται με **μία** αλλαγή:

| παραλλαγή | τι αλλάζει | ερώτημα |
|---|---|---|
| W0 / R0 | τίποτα | αναπαράγει το corpus; |
| W1 | σβήνονται οι λέξεις defect/cooperate από το δικό του μήνυμα | αντιγράφει τη λέξη ή τη στάση; |
| W2 | καθόλου δικό του μήνυμα (σιωπή) | το σημείο αναφοράς |
| S1 | το ίδιο κείμενο, ως μήνυμα γείτονα | μετρά ποιος το είπε; |
| R1 | σβήνονται οι λέξεις από τα ληφθέντα | λέξη ή πρόθεση; |
| D0–D2 | το κέντρο λαμβάνει 0, 1, 2 από 3 εχθρικά | δόση–απόκριση |

5 μοντέλα × 675 κλήσεις, χωρίς παιχνίδι. Οι κανόνες ανάγνωσης είναι στο
`docs/replay_plan.md`, γραμμένοι πριν από τις κλήσεις. Τα πλαίσια
(`replay/contexts.json`) είναι μέσα στο repo, άρα **δεν χρειάζεται dataset**.

## Setup

1. Settings → Accelerator → **GPU T4 x2** (δύο μοντέλα παράλληλα)
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach to notebook** (για Llama και τα δύο gemma)

Μετά **Save Version → Save & Run All**. Τίποτα δεν αλλάζει στο notebook.

| κάρτα | μοντέλα, με τη σειρά | ~ώρες |
|---|---|---|
| 0 | gemma-2-9b, gemma-2-2b | 3,2 |
| 1 | Qwen3-4B, Llama-3.1-8B, Qwen2.5-7B | 3,8 |

Οι ώρες βγαίνουν από τα μετρημένα δευτερόλεπτα ανά κλήση στα κελιά πηγής
βλάβης (TRACK_RECORD), ±30%. Αν κοπεί κάτι, το `replay.py` συνεχίζει από εκεί
που έμεινε όταν ξανατρέξει πάνω στο ίδιο αρχείο.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
assert N_GPU >= 2, ('Μία μόνο κάρτα: το notebook είναι υπολογισμένο για δύο μοντέλα '
                    'παράλληλα -- Settings -> Accelerator -> GPU T4 x2')

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

G2B, Q25 = 'google/gemma-2-2b-it', 'Qwen/Qwen2.5-7B-Instruct'
LLA, Q3, G9B = 'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen3-4B', 'google/gemma-2-9b-it'
LANES = ([G9B, G2B], [Q3, LLA, Q25])
HOURS = 4
CALLS = 675

for f in ('replay.py', 'replay_report.py', 'replay/contexts.json', 'docs/replay_plan.md'):
    assert os.path.exists(f), f'Λείπει το {f} -- κάνε git push πρώτα.'
for lane in LANES:
    for m in lane:
        p = f"pools/{m.split('/')[-1]}_cooperative_r1.json"
        assert os.path.exists(p), f'Λείπει η λίστα {p} -- κάνε git push πρώτα.'
assert 'Model loaded' in open('llm_client.py', encoding='utf-8').read(), \
    'το llm_client.py δεν τυπώνει «Model loaded» -- η κλειδαριά φόρτωσης δεν θα ξεκλείδωνε'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

## Ο έλεγχος που κρίνει το πείραμα — κοστίζει μηδέν

Για κάθε μοντέλο: 20 πλαίσια γράφοντα, 20 αναγνώστη και 20 σιωπής, και 675
κλήσεις. Μετά ξαναχτίζει ένα prompt γράφοντα με τον κώδικα που θα τρέξει και
δείχνει τι αλλάζει από παραλλαγή σε παραλλαγή: μόνο η γραμμή του δικού του
μηνύματος και οι γραμμές των ληφθέντων.

In [ ]:
import collections, json, sys
sys.path.insert(0, '.')
import replay
from engine import make_engine
from prompts import build_ct_action_user

ctx = json.load(open('replay/contexts.json', encoding='utf-8'))['contexts']
count = collections.Counter((c['model'], c['kind']) for c in ctx)
for lane in LANES:
    for m in lane:
        short = m.split('/')[-1]
        got = {k: count[(short, k)] for k in ('writer', 'reader', 'silence')}
        assert got == {'writer': 20, 'reader': 20, 'silence': 20}, f'{short}: {got}'
        pool = json.load(open(f'pools/{short}_cooperative_r1.json', encoding='utf-8'))['messages']
        n = sum(len(replay.variants(c, pool)) for c in ctx
                if c['model'] == short and c['kind'] in ('writer', 'reader'))
        assert n * 5 == CALLS, f'{short}: {n * 5} κλήσεις, όχι {CALLS}'
        print(f'{short:24s} 20/20/20 πλαίσια, {n * 5} κλήσεις')

c = next(c for c in ctx if c['model'] == 'Qwen3-4B' and c['kind'] == 'writer' and not c['hub'])
agent = make_engine(replay.to_config(c['cfg'])).build_agents(client=None, run_id=c['run_id'])[c['agent']]
pool = json.load(open('pools/Qwen3-4B_cooperative_r1.json', encoding='utf-8'))['messages']
base = None
for name, own, rec, _ in replay.variants(c, pool):
    user = build_ct_action_user(agent._history_text([]), 1, own, rec)
    if base is None:
        base = user.splitlines()
        print('W0, όλο το μήνυμα χρήστη:\n' + user)
        continue
    diff = [l for l in user.splitlines() if l not in base]
    print(f'{name}: ' + ' | '.join(l.strip() for l in diff))
    assert 1 <= len(diff) <= 2, f'{name}: άλλαξαν {len(diff)} γραμμές -- σταμάτα'
print('\nκάθε παραλλαγή αλλάζει μόνο ό,τι λέει το σχέδιο')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    raise SystemExit(f'Λείπει το HF_TOKEN (Llama και gemma είναι gated): {e}')

## Εκτέλεση: δύο κάρτες, μία ουρά η καθεμία

Κάθε μοντέλο είναι ένα `replay.py run` με δικό του log στο `/kaggle/working/logs/`
και δικό του αρχείο κλήσεων στο `/kaggle/working/replay_out/`. Τα μοντέλα
φορτώνουν ένα-ένα, και τα αρχεία κάθε μοντέλου σβήνονται από τον δίσκο όταν
τελειώσει. Κάθε 10 λεπτά τυπώνεται η πρόοδος.

In [ ]:
import glob, os, shutil, subprocess, sys, threading, time

LOG_DIR, OUT_DIR = '/kaggle/working/logs', '/kaggle/working/replay_out'
POLL_S, REPORT_S, DEADLINE_H = 30, 600, 11.5
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

try:
    from huggingface_hub import constants as _hfc
    HF_CACHE = _hfc.HF_HUB_CACHE
except Exception:
    HF_CACHE = os.path.expanduser('~/.cache/huggingface/hub')

def cache_dir(model):
    return os.path.join(HF_CACHE, 'models--' + model.replace('/', '--'))

def out_of(model):
    return os.path.join(OUT_DIR, model.split('/')[-1] + '.jsonl')

def calls_done(model):
    try:
        with open(out_of(model), encoding='utf-8') as fh:
            return sum(1 for _ in fh)
    except FileNotFoundError:
        return 0

def log_has(path, needle):
    try:
        with open(path, encoding='utf-8', errors='replace') as fh:
            return needle in fh.read()
    except FileNotFoundError:
        return False

load_lock = threading.Lock()   # ένα μοντέλο φορτώνει τη φορά
guard = threading.Lock()
state, finished, procs = {}, [], {}
t0 = time.time()

def run_lane(gpu, models):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2')
    for model in models:
        log = os.path.join(LOG_DIR, f"gpu{gpu}_{model.split('/')[-1]}.log")
        t = time.time()
        try:
            with load_lock:
                with guard:
                    state[gpu] = dict(model=model, log=log, t0=t, phase='φόρτωση')
                with open(log, 'w', encoding='utf-8') as fh:
                    proc = subprocess.Popen([sys.executable, 'replay.py', 'run', '--model', model,
                                             '--out', out_of(model)],
                                            stdout=fh, stderr=subprocess.STDOUT, env=env)
                with guard:
                    procs[gpu] = proc
                while proc.poll() is None and not log_has(log, '[local] Model loaded'):
                    time.sleep(10)
            with guard:
                state[gpu]['phase'] = 'κλήσεις'
            rc = proc.wait()
            n = calls_done(model)
            status = 'OK' if rc == 0 and n == CALLS else f'FAILED (exit {rc}, {n}/{CALLS})'
        except Exception as e:
            status = f'ERROR {e!r}'[:60]
        shutil.rmtree(cache_dir(model), ignore_errors=True)
        with guard:
            finished.append((gpu, model, status, (time.time() - t) / 60, log))
    with guard:
        state[gpu] = None

threads = []
for gpu, models in enumerate(LANES):
    th = threading.Thread(target=run_lane, args=(gpu, models), daemon=True)
    th.start()
    threads.append(th)
    time.sleep(2)

n_reported, last = 0, time.time()
while any(th.is_alive() for th in threads):
    time.sleep(POLL_S)
    if time.time() - t0 > DEADLINE_H * 3600:
        with guard:
            for p in procs.values():
                if p.poll() is None:
                    p.terminate()
    with guard:
        snap = {g: dict(s) for g, s in state.items() if s}
        done = list(finished)
    for g, model, st, mins, _ in done[n_reported:]:
        print(f"--> κάρτα {g} · {model.split('/')[-1]}: {st}, {mins:.0f} λεπτά", flush=True)
    n_reported = len(done)
    if time.time() - last >= REPORT_S:
        last = time.time()
        print(f'--- {(last - t0) / 3600:.1f} h ---', flush=True)
        for g, s in sorted(snap.items()):
            print(f"  κάρτα {g}: {s['model'].split('/')[-1]:24s} {s['phase']:8s} "
                  f"{calls_done(s['model']):4d}/{CALLS} κλήσεις, "
                  f"{(time.time() - s['t0']) / 3600:.1f} h", flush=True)

for th in threads:
    th.join()
for g, model, st, mins, _ in finished[n_reported:]:
    print(f"--> κάρτα {g} · {model.split('/')[-1]}: {st}, {mins:.0f} λεπτά")

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for g, model, st, mins, _ in finished:
    print(f"  κάρτα {g}  {st:28s} {mins:6.0f} λ   {model.split('/')[-1]}")
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h  (εκτίμηση ~{HOURS} h)')
for g, model, st, mins, log in finished:
    if st != 'OK':
        print('\n' + '-' * 70)
        print(f'ΑΠΟΤΥΧΙΑ: κάρτα {g} · {model} -- οι τελευταίες γραμμές του {log}:')
        with open(log, encoding='utf-8', errors='replace') as fh:
            print(''.join(fh.readlines()[-25:]))

## Ανάλυση με τους κανόνες του σχεδίου, και zip

Το `replay_report.py` εφαρμόζει τους κανόνες του `docs/replay_plan.md` όπως
είναι γραμμένοι: πρώτα αν το replay αναπαράγει το corpus, μετά αν υπάρχει βλάβη
στον 1ο γύρο, και μόνο τότε λέξη ή στάση, απόδοση, δόση. Το Holm τρέχει στα
πέντε μοντέλα, άρα οι ετυμηγορίες είναι οριστικές μόνο όταν έχουν τελειώσει όλα.

In [ ]:
import glob, os, shutil, subprocess, sys

outs = sorted(glob.glob(f'{OUT_DIR}/*.jsonl'))
print(f'{len(outs)} αρχεία κλήσεων')
r = subprocess.run([sys.executable, 'replay_report.py', '--results', *outs,
                    '--csv', f'{OUT_DIR}/replay_summary.csv'], capture_output=True, text=True)
print(r.stdout[-15000:]); print(r.stderr[-2000:])
with open(f'{OUT_DIR}/replay_report.txt', 'w', encoding='utf-8') as fh:
    fh.write(r.stdout)

shutil.copytree(LOG_DIR, os.path.join(OUT_DIR, 'logs'), dirs_exist_ok=True)
out = shutil.make_archive('/kaggle/working/replay_results', 'zip', OUT_DIR)
print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')

## Μετά

1. Κατέβασε το `replay_results.zip` και αποσυμπίεσέ το σε νέο φάκελο
   `diplomatikh/replay_runs/`. Δεν είναι runs παιχνιδιού: δεν περνά από το ingest.
2. Γράψε στο `TRACK_RECORD.md` τον χρόνο κάθε μοντέλου από τον απολογισμό.
3. Η ανάλυση ξανατρέχει τοπικά, ίδια:

```
python cheaptalk_bench/replay_report.py --results replay_runs/*.jsonl
```